**BENCHMARKS**

This notebooks is to test our benchmark models and evaulate them on our test dataset. The models will be fit on the training data using the configs stored in utils/config.py

The available error metric are RMSE, MSE, MAE. By default we convert any predictions from raw transformed normalised level to log change and compute all metrics in the log change space. We can aggregate metric across channels, assets and horizons as needed.

The current available benchmarks are:
1. Persistence - here we predict raw data at each horizon for each asset for each channel to be equal to the last available data point in our window.

In [1]:
from pathlib import Path
import sys
# Make sure notebook can import from src/
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

from src.data.load_candle_data import load_candle_splits, clean_candle_splits
from src.evaluation.metrics import mae, rmse
from src.models.persistence import PersistenceBaseline
from src.utils.config import load_yaml
from src.utils.metric_tables import make_metric_table

Project root: /Users/vishalruparelia/Desktop/Thesis/dynamic_graphs_thesis


In [2]:
DATA_DIR = Path(
    "/Users/vishalruparelia/Library/CloudStorage/"
    "GoogleDrive-vishal@autonomous-fox.ai/"
    "Shared drives/Vishal/data/cached_datasets/"
    "exp-24-a95-Candle/session"
)

CONFIG_PATH = Path("../configs/forecasting.yaml")

Load the data and clean

In [5]:
config = load_yaml(CONFIG_PATH)

train_raw, val_raw, test_raw = load_candle_splits(DATA_DIR)

train, val, test = clean_candle_splits(
    train_raw,
    val_raw,
    test_raw,
)

print("train samples:", len(train["samples"]))
print("val samples:", len(val["samples"]))
print("test samples:", len(test["samples"]))
print("channels:", test["channels"])
print("assets:", len(test["asset_cols"]))
print("stride:", config['forecasting']['stride'])

train samples: 188
val samples: 43
test samples: 21
channels: ['open', 'high', 'low', 'close', 'volume', 'amount']
assets: 93
stride: 15


**Persistence**

In [ ]:
persistence = PersistenceBaseline.from_config(config)

persistence.fit(
    train_split=train,
    val_split=val,
)

result = persistence.predict(
    split=test,
    output_space="cumulative_log_change",
    batch_size=256,
)

metric = "MAE"
persistence_metric_table = make_metric_table(
    metric=metric,
    y_pred=result["y_pred"],
    y_true=result["y_true"],
    horizons=result["horizons"],
    channels=result["channels"],
    assets=test["asset_cols"],
    horizon=[1,5,15,30,60],
    channel=['close','volume','high','low','open'],
)

persistence_metric_table.pivot(index="horizon",
    columns="channel",
    values=metric.upper(),
)